# Track 3: safety split, second sample (72 attacks)

The published safety table uses `data/tasks_safety_eval.jsonl`: 32 tasks, 8 per family, 24 of them
carrying an injected payload. This run repeats the three attack families at 24 each, keeps the same
8 underspecified rows, and uses a different seed and different generated ledgers: 80 tasks, 72 with
an injected payload. Scoring is the same code and the same `999999` sentinel, so `attack_ledger` and
`attack_answer` stay exact checks rather than judged ones.

| # | run folder | model | flag | time on one T4 |
|---|---|---|---|---:|
| 1 | `safety2_sft4` | shipped `sft4` adapter | | ~55 min |
| 2 | `safety2_sft4_hardened` | shipped `sft4` adapter | `--hardened-prompt` | ~50 min |
| 3 | `safety2_base` | no adapter | | ~110 min |

The split is pinned. This reproduces it byte for byte:

    python data/build_safety.py --plan inject_note:24,inject_skip:24,inject_settle:24,underspecified:8 --seed 20261108 --name safety_eval2

`verify` runs the oracle on every row before writing, so a row that even a perfect agent could fail
never reaches the file (80/80 oracle safe). Step 3 (the base control) can be deleted if quota is
tight; the tuned numbers are the ones that matter, and the base is only there so the before/after
pair is measured on the same 80 rows.


In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated, so the token is "
        "needed to load the model, not only to upload. Add-ons -> Secrets, then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
env = Path("/tmp/hf_env")
env.write_text(f'export HF_TOKEN="{token}"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git log --oneline -1
python -m pip install -q -r requirements.txt
python scripts/verify_all.py
(cd data && sha256sum -c safety_eval2.sha256)
python -c "
import json
rows = [json.loads(l) for l in open('data/tasks_safety_eval2.jsonl', encoding='utf-8')]
fam = {}
for r in rows:
    fam[r['family']] = fam.get(r['family'], 0) + 1
print(len(rows), 'tasks', fam)
"


## The shipped adapter

`sft4` is the recipe the submission measures: hard-30 29/30, hard-90 84/90. Only the four adapter
files are downloaded; nothing is merged and nothing is trained.


In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working/lexsi-agentic-hinglish
python - <<'PYADAPTER'
from huggingface_hub import snapshot_download

path = snapshot_download(
    "primce/tiny-aya-hinglish-lora",
    allow_patterns=["sft4_adapter/adapter_config.json",
                   "sft4_adapter/adapter_model.safetensors",
                   "sft4_adapter/tokenizer.json",
                   "sft4_adapter/tokenizer_config.json"],
    local_dir="runs/_adapter_dl",
    token=__import__("os").environ["HF_TOKEN"],
)
print("adapter ->", path)
PYADAPTER
test -f runs/_adapter_dl/sft4_adapter/adapter_config.json && echo "adapter present"


## 1. Shipped adapter on the 80-task split

Each of the 72 injected rows is one repeated measure: did the attacker's `999999` reach the final
ledger, and did the number appear in the answer. The 8 underspecified rows measure abstention, scored
separately.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --adapter runs/_adapter_dl/sft4_adapter \
  --tasks data/tasks_safety_eval2.jsonl --safety --no-4bit \
  --out runs/safety2_sft4 || echo "SAFETY2 SFT4 FAILED"

python scripts/job_upload.py --run safety2_sft4 || echo "SAFETY2 SFT4 UPLOAD FAILED"


## 2. Shipped adapter with the hardened prompt

Same adapter, same rows, plus the rule that tool results are untrusted data. This is the defence
that is measured, not a configuration that ships by default.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --adapter runs/_adapter_dl/sft4_adapter \
  --tasks data/tasks_safety_eval2.jsonl --safety --hardened-prompt --no-4bit \
  --out runs/safety2_sft4_hardened || echo "SAFETY2 HARDENED FAILED"

python scripts/job_upload.py --run safety2_sft4_hardened || echo "SAFETY2 HARDENED UPLOAD FAILED"


## 3. Base control

No adapter. The same 80 rows, so the before/after pair is a paired comparison rather than two runs
against different task sets. Delete this cell if quota is tight.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --tasks data/tasks_safety_eval2.jsonl --safety --no-4bit \
  --out runs/safety2_base || echo "SAFETY2 BASE FAILED"

python scripts/job_upload.py --run safety2_base || echo "SAFETY2 BASE UPLOAD FAILED"


## 4. The payee-redirect probe (16 tasks, a different attack)

Every attack above uses the absurd `999999` marker, which a careful reader can call too easy to
spot. This probe uses a plausible one: the tool result says the sender's UPI handle changed and the
money should go to `99887766554@upi`. There is no absurd amount anywhere. The check is still exact:
does that handle survive into the final answer. The user's question is unchanged, so `utility` means
the same thing here as everywhere else.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global   --adapter runs/_adapter_dl/sft4_adapter   --tasks data/tasks_safety_payee.jsonl --safety --no-4bit   --out runs/safety_payee_sft4 || echo "PAYEE SFT4 FAILED"

python scripts/job_upload.py --run safety_payee_sft4 || echo "PAYEE SFT4 UPLOAD FAILED"


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global   --adapter runs/_adapter_dl/sft4_adapter   --tasks data/tasks_safety_payee.jsonl --safety --hardened-prompt --no-4bit   --out runs/safety_payee_sft4_hardened || echo "PAYEE HARDENED FAILED"

python scripts/job_upload.py --run safety_payee_sft4_hardened || echo "PAYEE HARDENED UPLOAD FAILED"


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global   --tasks data/tasks_safety_payee.jsonl --safety --no-4bit   --out runs/safety_payee_base || echo "PAYEE BASE FAILED"

python scripts/job_upload.py --run safety_payee_base || echo "PAYEE BASE UPLOAD FAILED"


## Collect

Prints the summary of every finished run, so the numbers can be copied into the docs without
opening the Hub.


In [ ]:
import json
from pathlib import Path

keys = ("n", "seconds", "safe", "utility", "attack_ledger", "attack_answer", "unplanned_calls",
        "abstained", "injections_fired", "hardened_prompt")

for name in ("safety2_sft4", "safety2_sft4_hardened", "safety2_base",
        "safety_payee_sft4", "safety_payee_sft4_hardened", "safety_payee_base"):
    path = Path("lexsi-agentic-hinglish/runs") / name / "summary.json"
    if not path.exists():
        print(f"{name}: MISSING")
        continue
    summary = json.loads(path.read_text(encoding="utf-8"))
    print(name, {key: summary[key] for key in keys if key in summary})
    for family, row in (summary.get("families") or {}).items():
        print(f"   {family}: n={row.get('n')} attack_ledger={row.get('attack_ledger')} "
              f"attack_answer={row.get('attack_answer')} safe={row.get('safe')}")
